# 15 - Train a Max-over-n-Step DQN Model Offline

This notebook follows the same offline workflow as `02_train_offline_dqn.ipynb`, but trains with `MaxNStepDqnObjective`:

1. Load previously collected `Datastore` streams from the Hub.
2. Build a `DataLoader` that samples fixed-length sequences from those streams.
3. Assemble a `Model` with two action-value heads: `selector` (deployed policy) and `max_return`.
4. Train with `MaxNStepDqnObjective` and save with `push_model_to_hub`.

The n-step returns are the same as `14_train_offline_n_step_dqn.ipynb` (complete windows only; a `γ` of `0` finishes early). The extras unique to maximizing over n-steps: two heads (`selector` is the deployed policy, `max_return` is the value head), bootstrap delayed `max_return` at the action the *online* selector chooses at each endpoint, and train `max_return` on the **max** of the complete candidates in `horizons` (here `(1, 3, 5, 10)`). The selector trains on the one-step candidate. No TD(λ), no Watkins. `get_action` uses `action_head="selector"` only.

This is a short usage example, not a full experiment. Evaluate a saved checkpoint in `09_inference.ipynb`.


In [ ]:
import torch

from mouse_core import AdamW
from mouse_core.data import (
    DataLoader,
    Augmenter,
    Tokenizer,
    compose,
    load_stores_from_hub,
)
from mouse_core.objectives import MaxNStepDqnObjective
from mouse_core.models import Model, Polyak, push_model_to_hub
from mouse_core.models.backbone import Qwen3Backbone
from mouse_core.models.embedding import NumericEmbedder
from mouse_core.models.heads import DiscreteActionValueHead


DATASET_ID = "mouse-example-dataset"                        # Hugging Face dataset repo for load_stores_from_hub
MODEL_ID = "mouse-example-model-max-n-step-offline"       # Hugging Face model repo for push_model_to_hub
MAX_ACTIONS = 4                               # number of discrete actions predicted by each head
MAX_OBS_DISCRETE = 64                         # vocabulary size for discrete observations
HORIZONS = (1, 3, 5, 10)                      # n-step candidates; max-return trains on their max
MAX_RETURN_WEIGHT = 1.0                       # multiplier on the max-return loss
SEQUENCE_LENGTH = 512                         # replay sequence length sampled by DataLoader
BATCH_SIZE = 4                                # sequences per optimizer step
NUM_CYCLES = 2                               # outer train cycles (print cadence)
TRAIN_STEPS = 50                             # optimizer updates per cycle (passed to run_train)
POLYAK_TAU_HEADS = 0.0001                     # delayed Q-head interpolation (0 = frozen, 1 = copy of the online heads)
POLYAK_TAU_ENCODER = 0.01                     # delayed encoder interpolation
POLYAK_TAU_BACKBONE = 0.01                    # delayed backbone interpolation


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Load Data

`load_stores_from_hub` downloads the dataset snapshot and reconstructs the saved `Datastore` objects. Each returned store is one ordered environment stream.


In [ ]:
stores = load_stores_from_hub(repo_id=DATASET_ID, split='train', force_download=True)

## Data pipeline

`DataLoader` samples contiguous windows up to `sequence_length` (a max) from one or more datastores. Each sequence may be shorter than the max depending on where the window starts in the store.

Pipeline order: `augmenter → tokenizer → pack → embedder`.

| Stage | Role |
| --- | --- |
| **Augmenter** | `dict → dict` (`fields=` value transforms; `seed_field=` for shared draws within a `reseed` generation). Action permute sets `input_vector_field` / `output_vector_field` on `info_q_star` so Q* stays aligned. |
| **Tokenizer** | `dict → StepTokens` (`input_field` / `output_field`; `objective_fields=` is `action` / `reward` / `episode_done` / `task_done`; `grouping_field=`) |

Compose `train_transform = compose(augmenter, tokenizer)`.
`DataLoader(transform=train_transform)` maps each step and packs into a `TokenBatch`.
Live inference in `09_inference.ipynb` uses the tokenizer without the augmenter so chosen actions match the env.


In [ ]:
# Pipeline order: augmenter → tokenizer

augmenter = Augmenter(
    seed_field="task_index",
    fields=[
        {
            "type": "discrete",
            "input_field": "action",
            "input_vector_field": "info_q_star",
            "vocab_size": MAX_ACTIONS,
            "permute": True,
        },
        {
            "type": "discrete",
            "input_field": "observation",
            "vocab_size": MAX_OBS_DISCRETE,
            "permute": True,
        },
    ],
)

tokenizer = Tokenizer(
    input_fields=[
        {
            "type": "discrete",
            "input_field": "action",
        },
        {
            "type": "discrete",
            "input_field": "observation",
        },
        {
            "type": "fourier",
            "input_field": "reward",
        },
        {
            "type": "discrete",
            "input_field": "episode_done",
        },
        {
            "type": "learnable",
            "output_field": "value",
            "tokens": 1,
            "head_output": True,
        },
    ],
    objective_fields=[
        {
            "input_field": "action",
        },
        {
            "input_field": "reward",
        },
        {
            "input_field": "episode_done",
        },
        {
            "input_field": "task_done",
        },
    ],
    grouping_field="task_index",
)

train_transform = compose(augmenter, tokenizer)

loader = DataLoader(
    stores=stores,
    sequence_length=SEQUENCE_LENGTH,
    batch_size=BATCH_SIZE,
    transform=train_transform,
    prefetch=4,
    num_workers=0,
)


## Build The Model

A Mouse Core `Model` has three main pieces:

- `NumericEmbedder` maps a tokenized `TokenBatch` (modalities keyed by name; add `vocab_size` / `std` here; `fourier` / `continuous` also need `fourier_min` / `fourier_max`) into vectors.
- `Qwen3Backbone` processes those tokens with a transformer backbone. Three arguments are required and describe how it runs on this machine rather than what it is, so they are not saved with the model and `load_model` asks for them again: `train_kernel` for the uncached forward (`"flex"`, block-sparse FlexAttention), `decode_kernel` for cached decode (`"flex"`, paged FlexAttention) and `dtype` for the base weights (`torch.float32` so the whole backbone trains in fp32). `model.to(device)` only moves.
- Two `DiscreteActionValueHead`s: `selector` (deployed greedy policy) and `max_return` (trained on the max over horizon targets). `action_head="selector"` makes `get_action` read only the selector.

The backbone exposes `hidden_dim`, and the embedder and head use that same value so the pieces connect cleanly.

`NumericEmbedder` modality types used here:

- `discrete` for integer IDs such as actions, observations, and episode/task done codes.
- `fourier` for scalar numeric values such as rewards.
- `learnable` for the trailing `value` token (no step field; flagged `head_output: True` so Q is read from it).

`Model(...)` wraps the pieces behind a single forward call that returns predictions, objective data, and an optional cache.


In [ ]:
backbone = Qwen3Backbone(train_kernel="flex", decode_kernel="flex", dtype=torch.float32, pretrained="Qwen/Qwen3-0.6B")

encoder = NumericEmbedder(
    hidden_dim=backbone.hidden_dim,
    modalities=[
        {
            "type": "discrete",
            "field": "action",
            "vocab_size": MAX_ACTIONS,
            "std": 0.02,
            "positions": 1,
        },
        {
            "type": "discrete",
            "field": "observation",
            "vocab_size": MAX_OBS_DISCRETE,
            "std": 0.02,
            "positions": 1,
        },
        {
            "type": "fourier",
            "field": "reward",
            "std": 0.02,
            "positions": 1,
            "fourier_min": 0.01,
            "fourier_max": 10.0,
        },
        {
            "type": "discrete",
            "field": "episode_done",
            "vocab_size": 3,
            "std": 0.02,
            "positions": 1,
        },
        {
            "type": "learnable",
            "field": "value",
            "tokens": 1,
            "std": 0.02,
            "positions": 1,
        },
    ],
)

head_kwargs = dict(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
)

model = Model(
    encoder=encoder,
    backbone=backbone,
    heads={
        "selector": DiscreteActionValueHead(**head_kwargs),
        "max_return": DiscreteActionValueHead(**head_kwargs),
    },
    action_head="selector",
    reasoner=None,
    recurrence=None,
).train().to(device)
print(model)


## Training Phase

Each outer cycle runs `TRAIN_STEPS` optimizer updates via `run_train`. Mouse Core abstractions do most of the work:

1. `inputs, objective_data = loader.next_batch()` samples ragged step windows (up to `SEQUENCE_LENGTH`).
2. `model(inputs)` embeds the `TokenBatch`, runs the backbone with per-sequence causal attention/RoPE, and produces flat per-step head predictions.
3. `objective(objective_data, predictions, delayed_predictions)` computes the max-over-n-step DQN loss and metrics.
4. `AdamW` updates weights. The backbone, encoder, and heads are fp32 (`dtype=torch.float32`), so every update lands in fp32 with no master weights.
5. Delayed Q comes from the delayed model: `delayed_model = model.delayed_copy()` is a frozen copy of the online model — the fp32 encoder, backbone, and Q head are copied. After the online forward, `delayed_model(inputs)` runs the same `TokenBatch` through the delayed model under `torch.no_grad()`. `polyak.update(tau_heads=POLYAK_TAU_HEADS, tau_encoder=POLYAK_TAU_ENCODER, tau_backbone=POLYAK_TAU_BACKBONE)` interpolates each section toward the online model after the optimizer step: `0` keeps it frozen, `1` copies the online weights (no delay). Every interpolated parameter is fp32, so a small `tau` is never rounded away.

`MaxNStepDqnObjective` takes required `horizons` (`HORIZONS`) and `max_return_weight` (`MAX_RETURN_WEIGHT`). The n-step returns are the same as `NStepDqnObjective`. The extras are the two heads, the max over complete horizons, and the selector that picks the bootstrap action. It interprets `episode_done` and `task_done` (each `0`/`1`/`2`) through separate discount factors. Incomplete horizons are masked. A `γ` of `0` includes that reward and stops. The return never crosses a run break (`sequence_id` / `grouping_field`).


In [ ]:
optimizer = AdamW(model.parameters(), lr=1e-05, weight_decay=0.0, betas=(0.9, 0.95), eps=1e-08)
delayed_model = model.delayed_copy()
polyak = Polyak(model, delayed_model)
objective = MaxNStepDqnObjective(horizons=HORIZONS, max_return_weight=MAX_RETURN_WEIGHT, gamma_step=1.0, gamma_episode_terminal=1.0, gamma_episode_truncated=1.0, gamma_task_terminal=0.0, gamma_task_truncated=0.0, grouping_field="task_index")

def run_train(*, model: Model, delayed_model: Model, polyak: Polyak, optimizer: AdamW, objective: MaxNStepDqnObjective, loader: DataLoader, num_steps: int) -> tuple[torch.Tensor, dict[str, float]]:
    """Run ``num_steps`` optimizer steps on batches from ``loader``."""
    model.train()
    loss: torch.Tensor | None = None
    metrics: dict[str, float] = {}
    for _ in range(num_steps):
        inputs, objective_data = loader.next_batch()
        out = model(inputs)
        with torch.no_grad():
            delayed_out = delayed_model(inputs)
        loss, metrics = objective(objective_data.to(device), out.predictions, delayed_out.predictions)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        polyak.update(tau_heads=POLYAK_TAU_HEADS, tau_encoder=POLYAK_TAU_ENCODER, tau_backbone=POLYAK_TAU_BACKBONE)
    assert loss is not None
    return (loss, metrics)


## Run

Each of `NUM_CYCLES` cycles calls `run_train(num_steps=TRAIN_STEPS)`. Score the checkpoint later in `09_inference.ipynb`.


In [ ]:
for cycle in range(NUM_CYCLES):
    loss, metrics = run_train(model=model, delayed_model=delayed_model, polyak=polyak, optimizer=optimizer, objective=objective, loader=loader, num_steps=TRAIN_STEPS)
    print(f"cycle={cycle} train  loss={loss.item():.4f}  q_sel={metrics['q_selector_mean']:.3f}  q_max={metrics['q_max_return_mean']:.3f}")
loader.close()


## Push To The Hub

`push_model_to_hub` saves the model architecture and weights together. Later, `load_model` can reconstruct the full `Model` without repeating the embedder, backbone, and head definitions.


In [ ]:
model.eval().to("cpu")
url = push_model_to_hub(model=model, repo_id=MODEL_ID, private=False, clear=True)
print(f"Pushed to {url}")
